In [6]:
import pandas as pd
import json
import glob

In [21]:
pd.read_csv(r"/workspace/qgi\trojan 2.0\data\processed\The Fund For Peace\Fragile State Index\MAIN - State Fragility Index.csv")

,Country,Year,Total,S1: Demographic Pressures,S2: Refugees and IDPs,C3: Group Grievance,E3: Human Flight and Brain Drain,E2: Economic Inequality,E1: Economy,P1: State Legitimacy,P2: Public Services,P3: Human Rights,C1: Security Apparatus,C2: Factionalized Elites,X1: External Intervention
0,Sudan,2006-01-01 00:00:00,112.3,9.6,9.7,9.7,9.1,9.2,7.5,9.5,9.5,9.8,9.8,9.1,9.8
1,Congo Democratic Republic,2006-01-01 00:00:00,110.1,9.5,9.5,9.1,8.0,9.0,8.1,9.0,9.0,9.5,9.8,9.6,10.0
2,Cote d'Ivoire,2006-01-01 00:00:00,109.2,8.8,7.6,9.8,8.5,8.0,9.0,10.0,8.5,9.4,9.8,9.8,10.0
3,Iraq,2006-01-01 00:00:00,109.0,8.9,8.3,9.8,9.1,8.7,8.2,8.5,8.3,9.7,9.8,9.7,10.0
4,Zimbabwe,2006-01-01 00:00:00,108.9,9.7,8.9,8.5,9.0,9.2,9.8,8.9,9.5,9.5,9.4,8.5,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3165,Switzerland,2023,17.8,2.4,3.2,2.1,1.0,2.4,1.6,0.3,1.6,0.4,1.4,1.0,0.4
3166,New Zealand,2023,16.7,1.1,1.2,2.0,1.6,2.6,2.6,0.5,1.1,0.5,1.6,1.4,0.5
3167,Finland,2023,16.0,1.7,1.9,0.3,1.5,1.6,2.7,0.4,1.0,0.5,2.0,1.4,1.0
3168,Iceland,2023,15.7,1.5,1.5,0.5,1.6,1.5,2.6,0.4,0.9,0.4,0.4,1.8,2.6


In [19]:
def read_xlsx(file_path, sheet_name):
    return pd.read_excel(file_path, sheet_name = sheet_name)

def extract_data_from_source(config):
    source_folder = config["source"]["folder"]
    file_pattern  = config["source"]["file_pattern"]
    sheet_name    = config["source"]["sheet_name"]
    files         = glob.glob(f"{source_folder}\{file_pattern}")
    dataframes    = []

    for file in files:
        df = read_xlsx(file, sheet_name)
        dataframes.append(df)

    return dataframes

with open(r"/workspace/qgi\trojan 2.0\config\fsi.json", "r") as file:
    config = json.load(file)
    
def merge_dataframes(dataframes, config):
    # Extract main and sub-index columns
    main_index         = config["structure"]["columns"]["main_index"]
    sub_indexes        = config["structure"]["columns"]["sub_indexes"]
    columns_to_extract = ["Country", "Year", main_index] + sub_indexes
    
    # Extract the relevant columns from each dataframe and merge them row-wise
    transformed_data = pd.concat([df[columns_to_extract] for df in dataframes], ignore_index = True)
    
    return transformed_data


def save_csv(df, destination_folder, file_name_pattern, index_name):
    ## Fragile States Index instruction: convert "Year" to string and just the YYYY part of the timestamp
    #year      = df["Year"].iloc[0]
    file_name = file_name_pattern.format(index_name=index_name)
    path      = f"{destination_folder}{file_name}"
    df.to_csv(path, index = False)

def load_data_to_destination(df, config):
    index_name         = config["structure"]["columns"]["main_index"]
    destination_folder = config["destination"]["folder"]
    file_name_pattern  = config["destination"]["file_name_pattern"]
    
    save_csv(df, destination_folder, file_name_pattern, index_name)

# Extract
load_data_to_destination(merge_dataframes(extract_data_from_source(config), config), config)